In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, average_precision_score
import random
#import itertools
from PIL import Image
#import os
#import glob
from pathlib import Path
#import shutil
import matplotlib.pyplot as plt

In [2]:
# --- Configuration ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device type?")
print(device)
print("\n")

# Define the paths and parameters prompt 2
DATA_ROOT = './data'  # IMPORTANT: structure of the data root should be data-root> train-metadata.csv
IMAGE_SIZE = 256    #128
EMBEDDING_DIM = 128
MARGIN = 1.0
#BATCH_SIZE = 16
NUM_EPOCHS = 30
LEARNING_RATE = 1e-4

# Hyperparameters prompt 1
#NUM_SAMPLES = 1500  # Simulate a subset of the dataset
#VALIDATION_SIZE = 0.1
#TEST_SIZE = 0.1 #0.2
TRAIN_BATCH_SIZE = 32 #64
VAL_TEST_BATCH_SIZE = 512
#EMBEDDING_DIM = 128
#MARGIN = 1.0  # Triplet Loss margin
#LEARNING_RATE = 1e-4
#NUM_EPOCHS = 20  # Training epochs for the Siamese Network
#NUM_EPOCHS_CLASSIFIER = 10 # Training epochs for the final classifier

Device type?
cuda




In [3]:
# Set seeds for reproducibility
SEED = 48515739
random.seed(SEED)
np.random.seed(SEED)
#torch.manual_seed(SEED)
#torch.cuda.manual_seed_all(SEED)


#def seed_everything(seed=42):
#    #Sets seed for reproducibility.
#    random.seed(seed)
#    os.environ['PYTHONHASHSEED'] = str(seed)
#    np.random.seed(seed)
#    torch.manual_seed(seed)
#    if torch.cuda.is_available():
#        torch.cuda.manual_seed(seed)
#        torch.cuda.manual_seed_all(seed)
#        torch.backends.cudnn.deterministic = True
#        torch.backends.cudnn.benchmark = False

#seed_everything()


In [4]:
def split_data(data_root):
    """
    Fetches reference dataframe
    Splits data frame in 80/10/10 train/validation/test sets
    Oversamples the minority class to have equal numbers of each class in the train set
    Returns three dataframes: the train set, the validation set, the test set

    Image files are not manipulated as it would cause unnecessary overhead
    """
    data_dir = Path(data_root)

    # Fetch the image names and labels dataset and load to a dataframe
    data_df = pd.read_csv((data_dir / "train-metadata.csv"), index_col=0)

    # Get IDs and labels for dataset train/validation/test splitting
    # The isic_id is unique
    image_ids = data_df["isic_id"]
    labels = data_df["target"]

    # Split into train, validation and test sets
    # 80% of data to train, 10% to validate, 10% to test
    # Split train and validation/test
    train_ids, val_test_ids, train_labels, val_test_labels = train_test_split(
        image_ids, labels, test_size=0.2, stratify=labels, random_state=SEED
    )
    # Split validation and test
    val_ids, test_ids, val_labels, test_labels = train_test_split(
        val_test_ids, val_test_labels, test_size=0.5, stratify=val_test_labels, random_state=SEED
    )

    # Subset dataframe for train, validation and test
    # The isic_id column will be used to fetch the images when dataloading
    # The dataframe index is reset for ease of access at dataloading phase
    train_samples = data_df[data_df["isic_id"].isin(train_ids)].reset_index(drop=True)
    val_samples = data_df[data_df["isic_id"].isin(val_ids)].reset_index(drop=True)
    test_samples = data_df[data_df["isic_id"].isin(test_ids)].reset_index(drop=True)

    # Oversample the minority class in the training set
    # There will be an equal amount of rows for each class
    normal_samples_size = train_samples[train_samples["target"]== 0].shape[0]
    melanoma_sample = train_samples[train_samples["target"]== 1]
    oversample_sample = melanoma_sample.sample(n=normal_samples_size - melanoma_sample.shape[0], replace=True, random_state=SEED)

    # Concatenate the data and the oversaampled data into one dataframe
    train_samples = pd.concat([train_samples, oversample_sample], ignore_index=True)
    train_samples = train_samples.sample(frac=1).reset_index(drop=True)
    # Logic: We duplicate some of the image references in the training data 
    # label dataframe. Since the images will be transformed when loaded, this 
    # will augment the melanoma samples. We only add duplicated rows as this 
    # array is what gets iterated on by the Dataloader. There is no need to 
    # duplicate the image, that is useless use of memory. The augmented array 
    # is shuffled so that randomisation is ensured when dataloaders iterate 
    # the dataset.

    return train_samples, val_samples, test_samples

In [5]:
class SkinDataset(Dataset):
    """
    Custom Dataset class for ISIC images and labels.
    """
    def __init__(self, root_dir, items_df, transform:transforms.Compose=None):

        # get the image folder path
        self.image_dir = (Path(root_dir) / 'image')
        # get the labels dataframe
        self.items_df = items_df
        # Label names
        self.classes = ['normal', 'melanoma']

        # Standard image transformation to which we add supplied tranformations
        self.transform = transforms.Compose(
            (transform.transforms if transform else [])+
            #[transforms.ToPILImage(),
            [transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])]
            )

        self.len = self.items_df.shape[0] 

    def __len__(self):
        return self.len
    
    def __getitem__(self, idx):

        # Get image information from the dataframe
        item = self.items_df.iloc[idx]
        # Get image label
        label = item["target"]
        # Get image
        image_name = item["isic_id"]
        image = Image.open(self.image_dir / (image_name + ".jpg")).convert('RGB')
        # Transform image
        image = self.transform(image)
        
        return image, torch.tensor(label, dtype=torch.long)

In [5]:
class TripletDataset(Dataset):
    """
    Custom Dataset for generating (Anchor, Positive, Negative) triplets.
    """
    def __init__(self, root_dir, items_df, transform=None):
        #self.root_dir = root_dir
        #self.transform = transform
    
        # get the image folder path
        self.image_dir = (Path(root_dir) / 'image')
        # get the labels dataframe
        self.items_df = items_df
        # Label names
        self.classes = ['normal', 'melanoma']

        # Standard image transformation to which we add supplied tranformations
        self.transform = transforms.Compose(
            (transform.transforms if transform else [])+
            #[transforms.ToPILImage(),
            [transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])]
            )
            
        # Total number of unique images to iterate over
        self.len = self.items_df.shape[0] 

    def __len__(self):
        return self.len

    def __getitem__(self, index):
        # 1. Select Anchor (A)
        #anchor_path, anchor_class = self.all_paths[index]

        # Get image information from the dataframe
        anchor = self.items_df.iloc[index]
        # Get image label
        anchor_class = anchor["target"]
        # Get image
        anchor_name = anchor["isic_id"]
        anchor_image = Image.open(self.image_dir / (anchor_name + ".jpg")).convert('RGB')
        # Transform image
        anchor_image = self.transform(anchor_image)
        
        # 2. Select Positive (P)
        # Select an image from the same class as the anchor, but not the anchor itself
        try:
            positive = self.items_df[(self.items_df["isic_id"]!=anchor_name) & (self.items_df["target"]==anchor_class)].sample()
        except:
            # Handle edge case where only one image exists in the class (should not happen in real ISIC)
            positive = anchor

        # Get image
        positive_name = positive["isic_id"].item()
        positive_image = Image.open(self.image_dir / (positive_name + ".jpg")).convert('RGB')
        # Transform image
        positive_image = self.transform(positive_image)

        # 3. Select Negative (N)
        # Select a class different from the anchor class (binary case is simple)
        negative_class = 1 - anchor_class
        # Select a negative sample
        negative = self.items_df[self.items_df["target"]==negative_class].sample()
        # Get image
        negative_name = negative["isic_id"].item()
        negative_image = Image.open(self.image_dir / (negative_name + ".jpg")).convert('RGB')
        # Transform image
        negative_image = self.transform(negative_image)

        # Return triplet and the anchor's original label for verification/testing
        return anchor_image, positive_image, negative_image, anchor_class

class ClassificationNet(nn.Module):
    """
    A simple linear head trained on top of the fixed embeddings 
    for the final binary classification (Melanoma vs. Normal).
    """
    def __init__(self, embedding_dim):
        super(ClassificationNet, self).__init__()
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2) # Two classes: 0 (Normal) and 1 (Melanoma)
        )

    def forward(self, x):
        return self.classifier(x)

class EmbeddingNet(nn.Module):
    """Simple non-pretrained CNN to generate image embeddings."""
    def __init__(self, image_size=IMAGE_SIZE, out_dim=EMBEDDING_DIM):
        super(EmbeddingNet, self).__init__()
        
        # Output size after Conv1 (256->128) -> Conv2 (128->64) -> Conv3 (64->32) -> Conv4 (32->16)
        # Layer 1: Conv -> ReLU -> Pool
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)
        
        # Layer 2: Conv -> ReLU -> Pool
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)
        
        # Layer 3: Conv -> ReLU -> Pool
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2, 2)

        # Layer 4: Conv -> ReLU -> Pool
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        self.pool4 = nn.MaxPool2d(2, 2)
        
        # Calculate the size before the first linear layer
        # 256 -> 128 -> 64 -> 32 -> 16. The output size is 16x16 with 256 channels.
        self.fc_input_size = 256 * (image_size // 16) * (image_size // 16) # 256 * 16 * 16 = 65536

        # Fully Connected Layer to produce the embedding
        self.fc1 = nn.Linear(self.fc_input_size, 512)
        self.fc_out = nn.Linear(512, out_dim)

        # classification head
        self.classifier = ClassificationNet(out_dim)

    def forward(self, x):
        x = self.pool1(nn.functional.relu(self.bn1(self.conv1(x))))
        x = self.pool2(nn.functional.relu(self.bn2(self.conv2(x))))
        x = self.pool3(nn.functional.relu(self.bn3(self.conv3(x))))
        x = self.pool4(nn.functional.relu(self.bn4(self.conv4(x))))
        
        # Flatten the feature map
        x = x.view(x.size(0), -1) 
        
        x = nn.functional.relu(self.fc1(x))
        # Final embedding output
        x = self.fc_out(x)
        
        # L2-normalize the embedding vector
        x = nn.functional.normalize(x, p=2, dim=1)
        return x
    
    def classify(self, x):
        return self.classifier(x)

In [6]:
class EmbeddingNet(nn.Module):
    """Simple non-pretrained CNN to generate image embeddings."""
    def __init__(self, image_size=IMAGE_SIZE, out_dim=EMBEDDING_DIM):
        super(EmbeddingNet, self).__init__()
        
        # load a ResNet model
        resnet = models.resnet50()

        self.extractor = nn.Sequential(*list(resnet.children())[:-1])

        self.fc_out = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, out_dim)
        )
        
        # Calculate the size before the first linear layer
        # 256 -> 128 -> 64 -> 32 -> 16. The output size is 16x16 with 256 channels.
        #self.fc_input_size = 256 * (image_size // 16) * (image_size // 16) # 256 * 16 * 16 = 65536

        # Fully Connected Layer to produce the embedding
        #self.fc1 = nn.Linear(self.fc_input_size, 512)
        #self.fc_out = nn.Linear(512, out_dim)

        # classification head
        self.classifier = nn.Linear(out_dim, 2)

    def forward(self, x):
        # extract features
        x = self.extractor(x)
        # Flatten the feature map
        x = x.view(x.size(0), -1) 
        # Final embedding output
        x = self.fc_out(x)
        
        # L2-normalize the embedding vector
        #x = nn.functional.normalize(x, p=2, dim=1)
        return x
    
    def classify(self, x):
        return self.classifier(x)

In [7]:
class TripletLoss(nn.Module):
    """
    Triplet loss function based on the distance between embeddings.
    L(A, P, N) = max(0, ||f(A) - f(P)||^2 - ||f(A) - f(N)||^2 + margin)
    """
    def __init__(self, margin=1.0):
        super(TripletLoss, self).__init__()
        self.margin = margin
        self.p = 2  # L2 distance

    def forward(self, anchor, positive, negative):
        # Calculate squared L2 distance
        d_pos = nn.functional.pairwise_distance(anchor, positive, p=self.p)
        d_neg = nn.functional.pairwise_distance(anchor, negative, p=self.p)
        
        # Triplet loss formula
        loss = torch.relu(d_pos - d_neg + self.margin).mean()
        return loss

def get_pairwise_distances(embeddings):
    """Computes the squared Euclidean distance matrix."""
    dot_product = torch.matmul(embeddings, embeddings.T)
    square_norm = torch.diag(dot_product)
    distances = square_norm.unsqueeze(0) - 2.0 * dot_product + square_norm.unsqueeze(1)
    distances[distances < 0] = 0 # Ensure non-negative distances
    #return torch.sqrt(distances + 1e-8)#distances.sqrt()
    mask = (distances == 0.0).float()

    # use this mask to set indices with a value of 0 to eps
    distances += mask * 1e-8

    # now it is safe to get the square root
    distances = torch.sqrt(distances)

    # undo the trick for numerical stability
    distances *= (1.0 - mask)

    return distances

def euclidean_distance_matrix(x):
    """Efficient computation of Euclidean distance matrix
    Args:
        x: Input tensor of shape (batch_size, embedding_dim)
        
    Returns:
        Distance matrix of shape (batch_size, batch_size)
    """
    # step 1 - compute the dot product

    # shape: (batch_size, batch_size)
    dot_product = torch.mm(x, x.t())

    # step 2 - extract the squared Euclidean norm from the diagonal

    # shape: (batch_size,)
    squared_norm = torch.diag(dot_product)

    # step 3 - compute squared Euclidean distances

    # shape: (batch_size, batch_size)
    distance_matrix = squared_norm.unsqueeze(0) - 2 * dot_product + squared_norm.unsqueeze(1)

    # get rid of negative distances due to numerical instabilities
    distance_matrix = F.relu(distance_matrix)

    # step 4 - compute the non-squared distances
    
    # handle numerical stability
    # derivative of the square root operation applied to 0 is infinite
    # we need to handle by setting any 0 to eps
    return torch.sqrt(distance_matrix + 1e-8)
    mask = (distance_matrix == 0.0).float()

    # use this mask to set indices with a value of 0 to eps
    distance_matrix += mask * 1e-8

    # now it is safe to get the square root
    distance_matrix = torch.sqrt(distance_matrix)

    # undo the trick for numerical stability
    distance_matrix *= (1.0 - mask)

    return distance_matrix


def get_triplets(labels, distances):
    """
    Performs Batch-Hard Triplet Mining.
    For each anchor, finds the hardest positive and the hardest negative in the batch.
    """
    #batch_size = labels.size(0)
    
    # Create mask for positive and negative pairs
    labels_equal = (labels.unsqueeze(0) == labels.unsqueeze(1))
    
    # 1. Hardest Positive (Anchor-Positive distance should be maximized)
    # Mask to select only positive pairs (i.e., same label, excluding self-distance)
    positive_mask = labels_equal.triu(diagonal=1) | labels_equal.tril(diagonal=-1) 
    
    # Set non-positive distances to a very small number for maximization (finding the largest distance)
    anchor_positive_dist = distances * positive_mask.float()
    
    # Max distance per row (Anchor) is the hardest positive
    hardest_positive_dist, _ = anchor_positive_dist.max(dim=1, keepdim=True)
    
    # 2. Hardest Negative (Anchor-Negative distance should be minimized)
    # Mask to select only negative pairs (i.e., different label)
    negative_mask = ~labels_equal
    
    # Set non-negative distances to a very large number for minimization (finding the smallest distance)
    # We use a copy to avoid in-place modification of the original distances tensor
    anchor_negative_dist = distances.clone()
    anchor_negative_dist[~negative_mask] = float('inf')
    
    # Min distance per row (Anchor) is the hardest negative
    hardest_negative_dist, _ = anchor_negative_dist.min(dim=1, keepdim=True)

    return hardest_positive_dist, hardest_negative_dist

def get_triplet_mask(labels):
    """compute a mask for valid triplets
    Args:
        labels: Batch of integer labels. shape: (batch_size,)
    Returns:
        Mask tensor to indicate which triplets are actually valid. Shape: (batch_size, batch_size, batch_size)
        A triplet is valid if:
        `labels[i] == labels[j] and labels[i] != labels[k]`
        and `i`, `j`, `k` are different.
    """
    # step 1 - get a mask for distinct indices

    # shape: (batch_size, batch_size)
    indices_equal = torch.eye(labels.size()[0], dtype=torch.bool, device=labels.device)
    indices_not_equal = torch.logical_not(indices_equal)
    # shape: (batch_size, batch_size, 1)
    i_not_equal_j = indices_not_equal.unsqueeze(2)
    # shape: (batch_size, 1, batch_size)
    i_not_equal_k = indices_not_equal.unsqueeze(1)
    # shape: (1, batch_size, batch_size)
    j_not_equal_k = indices_not_equal.unsqueeze(0)
    # Shape: (batch_size, batch_size, batch_size)
    distinct_indices = torch.logical_and(torch.logical_and(i_not_equal_j, i_not_equal_k), j_not_equal_k)

    # step 2 - get a mask for valid anchor-positive-negative triplets

    # shape: (batch_size, batch_size)
    labels_equal = labels.unsqueeze(0) == labels.unsqueeze(1)
    # shape: (batch_size, batch_size, 1)
    i_equal_j = labels_equal.unsqueeze(2)
    # shape: (batch_size, 1, batch_size)
    i_equal_k = labels_equal.unsqueeze(1)
    # shape: (batch_size, batch_size, batch_size)
    valid_indices = torch.logical_and(i_equal_j, torch.logical_not(i_equal_k))

    # step 3 - combine two masks
    mask = torch.logical_and(distinct_indices, valid_indices)

    return mask

class BatchAllTtripletLoss(nn.Module):
    """Uses all valid triplets to compute Triplet loss
    Args:
        margin: Margin value in the Triplet Loss equation
    """
    def __init__(self, margin=1.):
        super().__init__()
        self.margin = margin
        
    def forward(self, embeddings, labels):
        """computes loss value.
        Args:
        embeddings: Batch of embeddings, e.g., output of the encoder. shape: (batch_size, embedding_dim)
        labels: Batch of integer labels associated with embeddings. shape: (batch_size,)
        Returns:
        Scalar loss value.
        """
        # step 1 - get distance matrix
        # shape: (batch_size, batch_size)
        distance_matrix = euclidean_distance_matrix(embeddings)

        # step 2 - compute loss values for all triplets by applying broadcasting to distance matrix

        # shape: (batch_size, batch_size, 1)
        anchor_positive_dists = distance_matrix.unsqueeze(2)
        # shape: (batch_size, 1, batch_size)
        anchor_negative_dists = distance_matrix.unsqueeze(1)
        # get loss values for all possible n^3 triplets
        # shape: (batch_size, batch_size, batch_size)
        triplet_loss = anchor_positive_dists - anchor_negative_dists + self.margin

        # step 3 - filter out invalid or easy triplets by setting their loss values to 0

        # shape: (batch_size, batch_size, batch_size)
        mask = get_triplet_mask(labels)
        triplet_loss *= mask
        # easy triplets have negative loss values
        triplet_loss = F.relu(triplet_loss)

        # step 4 - compute scalar loss value by averaging positive losses
        num_positive_losses = (triplet_loss > 1e-8).float().sum()
        triplet_loss = triplet_loss.sum() / (num_positive_losses + 1e-8)

        return triplet_loss
  

class TripletMarginLoss(nn.Module):
    """
    Combines Triplet Loss with Batch-Hard mining.
    """
    def __init__(self, margin):
        super(TripletMarginLoss, self).__init__()
        self.margin = margin
    
    def forward(self, embeddings, labels):
        # Get pairwise distances
        distances = get_pairwise_distances(embeddings)
        
        # Perform Batch-Hard mining to find the hardest (Ap) and (An) for each Anchor
        hardest_positive_dist, hardest_negative_dist = get_triplets(labels, distances)
        
        # Calculate Triplet Loss: max(0, d(a,p) - d(a,n) + margin)
        losses = torch.relu(hardest_positive_dist - hardest_negative_dist + self.margin)
        
        # Only consider anchors that had at least one valid hard positive and hard negative
        # In this Batch-Hard implementation, every anchor should theoretically have a pair 
        # as long as the batch is sampled to be balanced (which it is via the DataLoader shuffle).
        
        return losses.mean()

def train_embedding_net(model, train_loader, criterion, optimizer, epochs, device):
    """Trains the Siamese Embedding Network using Triplet Loss."""
    model.train()
    print("\n--- Training Embedding Network (Metric Learning) ---")
    
    for epoch in range(1, epochs + 1):
        running_loss = 0.0
        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            
            # Generate embeddings
            embeddings = model(images)
            
            # Calculate Triplet Loss using Batch-Hard mining
            loss = criterion(embeddings, labels)
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * len(images)
            
            if (i + 1) % 50 == 0:
                print(f'Epoch {epoch}/{epochs}, Batch {i+1}/{len(train_loader)}, Loss: {loss.item():.4f}')

        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {epoch} finished. Average Loss: {epoch_loss:.4f}")

    print("Embedding network training complete.")

def train_classifier_head(embedding_net, classifier_head, train_loader, criterion, optimizer, epochs, device):
    """Trains the Classification Head while freezing the Embedding Net."""
    embedding_net.eval()
    classifier_head.train()
    print("\n--- Training Classification Head ---")

    for epoch in range(1, epochs + 1):
        running_loss = 0.0
        correct_predictions = 0
        total_samples = 0
        
        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()

            # Generate embeddings (NO GRADIENT)
            with torch.no_grad():
                embeddings = embedding_net(images)
            
            # Classify
            outputs = classifier_head(embeddings)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            # Statistics
            running_loss += loss.item() * len(images)
            _, preds = torch.max(outputs, 1)
            correct_predictions += torch.sum(preds == labels.data).item()
            total_samples += len(images)

        epoch_loss = running_loss / total_samples
        epoch_acc = correct_predictions / total_samples
        print(f"Epoch {epoch} finished. Avg Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")


In [9]:
def evaluate_model(embedding_net, embedding_crit, classifier_crit, evaluation_loader, device):
    """Evaluates the final model on the test set."""
    embedding_net.eval()
    
    all_labels = []
    all_predictions = []
    all_probs = []
    emb_running_loss = 0.0
    clas_running_loss = 0.0
    total_samples = 0
    
    with torch.no_grad():
        for images, labels in evaluation_loader:
            images, labels = images.to(device), labels.to(device)
            
            embeddings = embedding_net(images)
            #outputs = embedding_net.classify(embeddings)

            emb_loss = embedding_crit(embeddings, labels)
            emb_running_loss += emb_loss.item() * len(images)

            #clas_loss = classifier_crit(outputs, labels)
            #clas_running_loss += clas_loss.item() * len(images)
            
            # Predictions and Probabilities
            #_, preds = torch.max(outputs, 1)
            #probs = torch.softmax(outputs, dim=1)[:, 1] # Probability of class 1 (Melanoma)

            #all_labels.extend(labels.cpu().numpy())
            #all_predictions.extend(preds.cpu().numpy())
            #all_probs.extend(probs.cpu().numpy())

            total_samples += len(images)

    emb_epoch_loss = emb_running_loss / total_samples
    #clas_epoch_loss = clas_running_loss / total_samples
    #acc = accuracy_score(all_labels, all_predictions)
    #try:
        # AUC is critical for imbalanced data like ISIC
        #auc = roc_auc_score(all_labels, all_probs)
    #except ValueError:
        # Handle cases where only one class is present (unlikely with stratify, but possible with small batches)
        #auc = 0.5 

    return emb_epoch_loss#, clas_epoch_loss, acc, auc
    #print("\n--- Final Test Set Results ---")
    #print(f"Overall Classification Accuracy: {overall_acc:.4f}")
    #print(f"ROC AUC Score (Melanoma): {overall_auc:.4f}")


    
    # We target an accuracy of around 0.8
    #if overall_acc >= 0.78:
    #    print("\n✅ Target Accuracy Achieved!")
    #else:
    #    print("\n⚠️ Target Accuracy Not Reached in Simulation. Increase epochs or adjust hyperparameters.")


In [10]:
def plot_logs(
        emb_train_loss_log, 
        clas_train_loss_log,
        train_accuracy_log,
        emb_val_loss_log,
        clas_val_loss_log,
        val_accuracy_log,
        val_ROC_AUC_log,
        epochs):
    
    plt.figure(figsize=(15, 15))

    plt.subplot(2, 2, 1)
    plt.plot(range(epochs), emb_train_loss_log, label='Train Loss', color='#F05039')
    plt.plot(range(epochs), emb_val_loss_log, label='Validation Loss', color='#3D65A5')
    plt.title('Embedding Loss over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(2, 2, 2)
    plt.plot(range(epochs), clas_train_loss_log, label='Train Loss', color='#F05039')
    plt.plot(range(epochs), clas_val_loss_log, label='Validation Loss', color='#3D65A5')
    plt.title('Classification Loss over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(2, 2, 3)
    plt.plot(range(epochs), train_accuracy_log, label='Train Accuracy', color='#F05039')
    plt.plot(range(epochs), val_accuracy_log, label='Validation Accuracy', color='#3D65A5')
    plt.title('Accuracy over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(2, 2, 4)
    plt.plot(range(epochs), val_ROC_AUC_log, label='ROC AUC', color='#3D65A5')
    plt.title('Validation ROC AUC over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('ROC AUC')
    plt.legend()

    plt.tight_layout()
    plt.savefig('training_logs.png')
    plt.show()
    #plt.close()

def train_nets(
        embedding_net, classifier_net, 
        train_loader, val_loader, 
        embedding_crit, classifier_crit, 
        embedding_opt, classifier_opt, 
        scheduler,
        epochs, 
        device):
    """Trains the Siamese Embedding Network using Triplet Loss."""
    
    print("\n--- Training Networks ---")

    # metric logging intialisation
    best_val_ROC_AUC = -1.0
    emb_train_loss_log = []
    clas_train_loss_log = []
    train_accuracy_log = []
    emb_val_loss_log = []
    clas_val_loss_log = []
    val_accuracy_log = []
    val_ROC_AUC_log = []
    
    for epoch in range(1, epochs + 1):
        embedding_net.train()
        classifier_net.train()
        emb_running_loss = 0.0
        clas_running_loss = 0.0
        correct_predictions = 0
        total_samples = 0

        print(f"\n==== Training Epoch {epoch} ====")

        # --- Training phase ----

        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            print(i)

            # ---- Embedding model training ----
            embedding_opt.zero_grad()
            
            # Generate embeddings
            embeddings = embedding_net(images)
            
            # Calculate Triplet Loss using Batch-Hard mining
            emb_loss = embedding_crit(embeddings, labels)
            
            emb_loss.backward()
            embedding_opt.step()
            
            emb_running_loss += emb_loss.item() * len(images)
            
            if (i + 1) % 50 == 0:
                print(f'Epoch {epoch}/{epochs}, Batch {i+1}/{len(train_loader)}, Embedding training loss: {emb_loss.item():.4f}')

            # ---- Classification model training ----
            classifier_opt.zero_grad()
            
            # Classify
            outputs = classifier_net(embeddings)
            clas_loss = classifier_crit(outputs, labels)
            print(clas_loss)
            
            clas_loss.backward()
            classifier_opt.step()
            
            # Statistics
            clas_running_loss += clas_loss.item() * len(images)
            _, preds = torch.max(outputs, 1)
            correct_predictions += torch.sum(preds == labels.data).item()
            total_samples += len(images)

        # embedding training epoch loss
        emb_epoch_loss = emb_running_loss / total_samples #len(train_loader.dataset)
        print(f"Epoch {epoch} finished. \nAverage Training Embedding Loss: {emb_epoch_loss:.4f}")

        # classification training epoch loss
        clas_epoch_loss = clas_running_loss / total_samples
        epoch_acc = correct_predictions / total_samples
        print(f"Average Training Classification Loss: {clas_epoch_loss:.4f}")
        print(f"Training Classification Accuracy: {epoch_acc:.4f}")

        # ---- Evaluation phase ----

        print("--- Validation phase ---")
        val_emb_loss, val_clas_loss, epoch_val_accuracy, epoch_val_ROC_AUC = evaluate_model(embedding_net, classifier_net, embedding_crit, classifier_crit, val_loader, device)
        print(f"Average Validation Embedding Loss: {val_emb_loss:.4f}")
        print(f"Average Validation Classification Loss: {val_clas_loss:.4f}")
        print(f"Validation Classification Accuracy: {epoch_val_accuracy:.4f}")
        print(f"Validation ROC AUC: {epoch_val_ROC_AUC:.4f}")

        # metric logging for plotting
        emb_train_loss_log.append(emb_epoch_loss)
        clas_train_loss_log.append(clas_epoch_loss)
        train_accuracy_log.append(epoch_acc)
        emb_val_loss_log.append(val_emb_loss)
        clas_val_loss_log.append(val_clas_loss)
        val_accuracy_log.append(epoch_val_accuracy)
        val_ROC_AUC_log.append(epoch_val_ROC_AUC)

        scheduler.step(epoch_val_ROC_AUC)

        # save best model based on ROC AUC
        if epoch_val_ROC_AUC > best_val_ROC_AUC:
            print(f"Previous best ROC AUC: {best_val_ROC_AUC:.4f}")
            best_val_ROC_AUC = epoch_val_ROC_AUC
            # Save model checkpoint
            print("Saving best model...")
            torch.save(embedding_net.state_dict(), Path(DATA_ROOT / 'best_embedding_model.pth'))
            torch.save(classifier_net.state_dict(), Path(DATA_ROOT / 'best_classifier_model.pth'))


    print("Network training complete.")

    # Graphical display of metric logs
    plot_logs(
        emb_train_loss_log, 
        clas_train_loss_log,
        train_accuracy_log,
        emb_val_loss_log,
        clas_val_loss_log,
        val_accuracy_log,
        val_ROC_AUC_log,
        epochs)

def train_nets(
        embedding_net, classifier_net, 
        train_loader, val_loader, 
        embedding_crit, classifier_crit, 
        embedding_opt, classifier_opt, 
        scheduler,
        epochs, 
        device):
    """Trains the Siamese Embedding Network using Triplet Loss."""
    
    print("\n--- Training Networks ---")

    # metric logging intialisation
    best_val_ROC_AUC = -1.0
    emb_train_loss_log = []
    clas_train_loss_log = []
    train_accuracy_log = []
    emb_val_loss_log = []
    clas_val_loss_log = []
    val_accuracy_log = []
    val_ROC_AUC_log = []
    
    for epoch in range(1, epochs + 1):
        embedding_net.train()
        classifier_net.train()
        emb_running_loss = 0.0
        clas_running_loss = 0.0
        correct_predictions = 0
        total_samples = 0

        print(f"\n==== Training Epoch {epoch} ====")

        # --- Training phase ----

        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            #print(i)

            # ---- Embedding model training ----
            
            
            # Generate embeddings
            embeddings = embedding_net(images)
            
            # Calculate Triplet Loss using Batch-Hard mining
            emb_loss = embedding_crit(embeddings, labels)
            
            # ---- Classification model training ----
            
            
            # Classify
            outputs = classifier_net(embeddings)
            clas_loss = classifier_crit(outputs, labels)

            # Statistics
            emb_running_loss += emb_loss.item() * len(images)
            clas_running_loss += clas_loss.item() * len(images)
            _, preds = torch.max(outputs, 1)
            correct_predictions += torch.sum(preds == labels.data).item()
            total_samples += len(images)

            if (i + 1) % 50 == 0:
                print(f'Epoch {epoch}/{epochs}, Batch {i+1}/{len(train_loader)}, Embedding training loss: {(emb_loss.item()):.4f}, Classification training loss: {(clas_loss.item()):.4f}')
            #print(clas_loss)

            total_loss = emb_loss + clas_loss

            embedding_opt.zero_grad()
            classifier_opt.zero_grad()
            total_loss.backward()
            embedding_opt.step()
            classifier_opt.step()

        # embedding training epoch loss
        emb_epoch_loss = emb_running_loss / total_samples #len(train_loader.dataset)
        print(f"Epoch {epoch} finished. \nAverage Training Embedding Loss: {emb_epoch_loss:.4f}")

        # classification training epoch loss
        clas_epoch_loss = clas_running_loss / total_samples
        epoch_acc = correct_predictions / total_samples
        print(f"Average Training Classification Loss: {clas_epoch_loss:.4f}")
        print(f"Training Classification Accuracy: {epoch_acc:.4f}")

        # ---- Evaluation phase ----

        print("--- Validation phase ---")
        val_emb_loss, val_clas_loss, epoch_val_accuracy, epoch_val_ROC_AUC = evaluate_model(embedding_net, classifier_net, embedding_crit, classifier_crit, val_loader, device)
        print(f"Average Validation Embedding Loss: {val_emb_loss:.4f}")
        print(f"Average Validation Classification Loss: {val_clas_loss:.4f}")
        print(f"Validation Classification Accuracy: {epoch_val_accuracy:.4f}")
        print(f"Validation ROC AUC: {epoch_val_ROC_AUC:.4f}")

        # metric logging for plotting
        emb_train_loss_log.append(emb_epoch_loss)
        clas_train_loss_log.append(clas_epoch_loss)
        train_accuracy_log.append(epoch_acc)
        emb_val_loss_log.append(val_emb_loss)
        clas_val_loss_log.append(val_clas_loss)
        val_accuracy_log.append(epoch_val_accuracy)
        val_ROC_AUC_log.append(epoch_val_ROC_AUC)

        scheduler.step(epoch_val_ROC_AUC)

        # save best model based on ROC AUC
        if epoch_val_ROC_AUC > best_val_ROC_AUC:
            print(f"Previous best ROC AUC: {best_val_ROC_AUC:.4f}")
            best_val_ROC_AUC = epoch_val_ROC_AUC
            # Save model checkpoint
            print("Saving best model...")
            torch.save(embedding_net.state_dict(), Path(DATA_ROOT / 'best_embedding_model.pth'))
            torch.save(classifier_net.state_dict(), Path(DATA_ROOT / 'best_classifier_model.pth'))


    print("Network training complete.")

    # Graphical display of metric logs
    plot_logs(
        emb_train_loss_log, 
        clas_train_loss_log,
        train_accuracy_log,
        emb_val_loss_log,
        clas_val_loss_log,
        val_accuracy_log,
        val_ROC_AUC_log,
        epochs)

In [11]:
def train_nets(
        train_loader, val_loader,
        embedding_net, embedding_opt, 
        embedding_crit, classifier_crit,
        scheduler,
        epochs, 
        device):
    """Trains the Siamese Embedding Network using Triplet Loss."""
    
    print("\n--- Training Networks ---")

    # metric logging intialisation
    best_val_ROC_AUC = -1.0
    emb_train_loss_log = []
    clas_train_loss_log = []
    train_accuracy_log = []
    emb_val_loss_log = []
    clas_val_loss_log = []
    val_accuracy_log = []
    val_ROC_AUC_log = []
    
    for epoch in range(1, epochs + 1):
        embedding_net.train()
        #classifier_net.train()
        emb_running_loss = 0.0
        clas_running_loss = 0.0
        correct_predictions = 0
        total_samples = 0

        print(f"\n==== Training Epoch {epoch} ====")
        #with torch.autograd.detect_anomaly():
        # --- Training phase ----

        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            # ---- Embedding model training ----
            embedding_opt.zero_grad()
            
            # Generate embeddings
            embeddings = embedding_net(images)
            
            # Calculate Triplet Loss using Batch-Hard mining
            emb_loss = embedding_crit(embeddings, labels)
            
            # ---- Classification model training ----
            
            
            # Classify
            #outputs = embedding_net.classify(embeddings)
            #clas_loss = classifier_crit(outputs, labels)

            # Statistics
            emb_running_loss += emb_loss.item() * len(images)
            #clas_running_loss += clas_loss.item() * len(images)
            #_, preds = torch.max(outputs, 1)
            #correct_predictions += torch.sum(preds == labels.data).item()
            total_samples += len(images)

            #print(emb_loss.item(), emb_running_loss, clas_loss.item(), clas_running_loss)
            if (i + 1) % 200 == 0:
                print(f'Epoch {epoch}/{epochs}, Batch {i+1}/{len(train_loader)}, Embedding training loss: {(emb_loss.item()):.4f}')#, Classification training loss: {(clas_loss.item()):.4f}')
                scheduler.step()
            #print(clas_loss)

            total_loss = emb_loss #+ clas_loss
            
            total_loss.backward()
            embedding_opt.step()

        # embedding training epoch loss
        emb_epoch_loss = emb_running_loss / total_samples #len(train_loader.dataset)
        print(f"Epoch {epoch} finished. \nAverage Training Embedding Loss: {emb_epoch_loss:.4f}")

        # classification training epoch loss
        #clas_epoch_loss = clas_running_loss / total_samples
        #epoch_acc = correct_predictions / total_samples
        #print(f"Average Training Classification Loss: {clas_epoch_loss:.4f}")
        #print(f"Training Classification Accuracy: {epoch_acc:.4f}")

        # ---- Evaluation phase ----

        print("--- Validation phase ---")
        val_emb_loss = evaluate_model(embedding_net, embedding_crit, classifier_crit, val_loader, device)
        #val_emb_loss, val_clas_loss, epoch_val_accuracy, epoch_val_ROC_AUC = evaluate_model(embedding_net, embedding_crit, classifier_crit, val_loader, device)
        print(f"Average Validation Embedding Loss: {val_emb_loss:.4f}")
        #print(f"Average Validation Classification Loss: {val_clas_loss:.4f}")
        #print(f"Validation Classification Accuracy: {epoch_val_accuracy:.4f}")
        #print(f"Validation ROC AUC: {epoch_val_ROC_AUC:.4f}")

        # metric logging for plotting
        emb_train_loss_log.append(emb_epoch_loss)
        #clas_train_loss_log.append(clas_epoch_loss)
        #train_accuracy_log.append(epoch_acc)
        emb_val_loss_log.append(val_emb_loss)
        #clas_val_loss_log.append(val_clas_loss)
        #val_accuracy_log.append(epoch_val_accuracy)
        #val_ROC_AUC_log.append(epoch_val_ROC_AUC)

        #scheduler.step(emb_epoch_loss)

        # save best model based on ROC AUC
        #if epoch_val_ROC_AUC > best_val_ROC_AUC:
        #    print(f"Previous best ROC AUC: {best_val_ROC_AUC:.4f}")
        #    best_val_ROC_AUC = epoch_val_ROC_AUC
            # Save model checkpoint
        #    print("Saving best model...")
        #    torch.save(embedding_net.state_dict(), (Path(DATA_ROOT) / 'best_embedding_model.pth'))


    print("Network training complete.")

    # Graphical display of metric logs
    plot_logs(
        emb_train_loss_log, 
        clas_train_loss_log,
        train_accuracy_log,
        emb_val_loss_log,
        clas_val_loss_log,
        val_accuracy_log,
        val_ROC_AUC_log,
        epochs)
    

def train_nets(
        embedding_net, classifier_net, 
        train_loader, val_loader, 
        embedding_crit, classifier_crit, 
        embedding_opt, classifier_opt, 
        scheduler,
        epochs, 
        device):
    """Trains the Siamese Embedding Network using Triplet Loss."""
    
    print("\n--- Training Networks ---")

    # metric logging intialisation
    best_val_ROC_AUC = -1.0
    emb_train_loss_log = []
    clas_train_loss_log = []
    train_accuracy_log = []
    emb_val_loss_log = []
    clas_val_loss_log = []
    val_accuracy_log = []
    val_ROC_AUC_log = []
    
    for epoch in range(1, epochs + 1):
        embedding_net.train()
        classifier_net.train()
        emb_running_loss = 0.0
        clas_running_loss = 0.0
        correct_predictions = 0
        total_samples = 0

        print(f"\n==== Training Epoch {epoch} ====")

        # --- Training phase ----

        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            print(i)

            # ---- Embedding model training ----
            
            embedding_opt.zero_grad()
            classifier_opt.zero_grad()
            
            # Generate embeddings
            embeddings = embedding_net(images)
            
            # Calculate Triplet Loss using Batch-Hard mining
            emb_loss = embedding_crit(embeddings, labels).detach()
            
            
            
            emb_running_loss += emb_loss.item() * len(images)
            
            

            # ---- Classification model training ----

            
            
            # Classify
            outputs = classifier_net(embeddings)
            clas_loss = classifier_crit(outputs, labels)

            if (i + 1) % 50 == 0:
                print(f'Epoch {epoch}/{epochs}, Batch {i+1}/{len(train_loader)}, Embedding training loss: {emb_loss.item():.4f}, Classification training loss: {clas_loss.item():.4f}')
            #print(clas_loss)
            
            emb_loss.backward()
            embedding_opt.step()
            embedding_opt.zero_grad()
            clas_loss.backward()
            classifier_opt.step()
            
            # Statistics
            clas_running_loss += clas_loss.item() * len(images)
            _, preds = torch.max(outputs, 1)
            correct_predictions += torch.sum(preds == labels.data).item()
            total_samples += len(images)

        # embedding training epoch loss
        emb_epoch_loss = emb_running_loss / total_samples #len(train_loader.dataset)
        print(f"Epoch {epoch} finished. \nAverage Training Embedding Loss: {emb_epoch_loss:.4f}")

        # classification training epoch loss
        clas_epoch_loss = clas_running_loss / total_samples
        epoch_acc = correct_predictions / total_samples
        print(f"Average Training Classification Loss: {clas_epoch_loss:.4f}")
        print(f"Training Classification Accuracy: {epoch_acc:.4f}")

        # ---- Evaluation phase ----

        print("--- Validation phase ---")
        val_emb_loss, val_clas_loss, epoch_val_accuracy, epoch_val_ROC_AUC = evaluate_model(embedding_net, classifier_net, embedding_crit, classifier_crit, val_loader, device)
        print(f"Average Validation Embedding Loss: {val_emb_loss:.4f}")
        print(f"Average Validation Classification Loss: {val_clas_loss:.4f}")
        print(f"Validation Classification Accuracy: {epoch_val_accuracy:.4f}")
        print(f"Validation ROC AUC: {epoch_val_ROC_AUC:.4f}")

        # metric logging for plotting
        emb_train_loss_log.append(emb_epoch_loss)
        clas_train_loss_log.append(clas_epoch_loss)
        train_accuracy_log.append(epoch_acc)
        emb_val_loss_log.append(val_emb_loss)
        clas_val_loss_log.append(val_clas_loss)
        val_accuracy_log.append(epoch_val_accuracy)
        val_ROC_AUC_log.append(epoch_val_ROC_AUC)

        scheduler.step(epoch_val_ROC_AUC)

        # save best model based on ROC AUC
        if epoch_val_ROC_AUC > best_val_ROC_AUC:
            print(f"Previous best ROC AUC: {best_val_ROC_AUC:.4f}")
            best_val_ROC_AUC = epoch_val_ROC_AUC
            # Save model checkpoint
            print("Saving best model...")
            torch.save(embedding_net.state_dict(), Path(DATA_ROOT / 'best_embedding_model.pth'))
            torch.save(classifier_net.state_dict(), Path(DATA_ROOT / 'best_classifier_model.pth'))


    print("Network training complete.")

    # Graphical display of metric logs
    plot_logs(
        emb_train_loss_log, 
        clas_train_loss_log,
        train_accuracy_log,
        emb_val_loss_log,
        clas_val_loss_log,
        val_accuracy_log,
        val_ROC_AUC_log,
        epochs)

train_samples, val_samples, test_samples = split_data(DATA_ROOT)

train_dataset = SkinDataset(DATA_ROOT, 
                            train_samples,
                            transform=transforms.Compose([
                                transforms.RandomRotation(degrees=10, fill=(255, 255, 255)),
                                transforms.RandomHorizontalFlip(p=0.5),
                                transforms.RandomVerticalFlip(p=0.5),
                                transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05)
                                ]))

val_dataset = SkinDataset(DATA_ROOT, val_samples, transform=None)

# Use standard DataLoader; Triplet mining is handled in the custom loss
train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, drop_last=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=VAL_TEST_BATCH_SIZE, shuffle=True, num_workers=0)

# Model Setup

# Embedding Net with Triplet Loss
embedding_net = EmbeddingNet(image_size=IMAGE_SIZE, out_dim=EMBEDDING_DIM).to(device)
triplet_criterion = BatchAllTtripletLoss(margin=MARGIN)
embedding_optimizer = optim.Adam(embedding_net.parameters(), lr=1e-5)
embedding_scheduler = optim.lr_scheduler.OneCycleLR(embedding_optimizer, max_lr=1e-5, steps_per_epoch=8,epochs=7, anneal_strategy="cos")
#embedding_scheduler = optim.lr_scheduler.CyclicLR(embedding_optimizer, base_lr = 1e-8, max_lr = 1e-5, step_size_up = 4,mode = "exp_range")
#embedding_scheduler = optim.lr_scheduler.ReduceLROnPlateau(embedding_optimizer, mode='min', factor=0.5, patience=5)

# Classification Head with Cross-Entropy Loss
#classifier_head = ClassificationNet(EMBEDDING_DIM).to(device)
classification_criterion = nn.CrossEntropyLoss().to(device)
#classifier_optimizer = optim.Adam(classifier_head.parameters(), lr=LEARNING_RATE)# * 5) # Faster learning rate for small head

train_nets(
        train_loader, val_loader,
        embedding_net, embedding_optimizer, 
        triplet_criterion, classification_criterion, 
        embedding_scheduler,
        7, 
        device)


def train_epoch(model, dataloader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    for img_a, img_p, img_n, _ in dataloader:
        img_a, img_p, img_n = img_a.to(device), img_p.to(device), img_n.to(device)

        optimizer.zero_grad()
        
        # Get embeddings
        emb_a = model(img_a)
        emb_p = model(img_p)
        emb_n = model(img_n)
        
        # Calculate loss
        loss = criterion(emb_a, emb_p, emb_n)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * img_a.size(0)

    return total_loss / len(dataloader.dataset)

In [14]:
def train_epoch(model, dataloader, criterion, classification_crit, optimizer, scheduler, device):
    model.train()
    #total_loss = 0.0
    all_labels = []
    all_predictions = []
    all_probs = []
    emb_running_loss = 0.0
    class_running_loss = 0.0
    total_samples = 0
    #print("la")
    for i, (img_a, img_p, img_n, label_a) in enumerate(dataloader):

        #print("yo", i+1)
        img_a, img_p, img_n, label_a = img_a.to(device), img_p.to(device), img_n.to(device), label_a.to(device)

        optimizer.zero_grad()
        
        # Get embeddings
        emb_a = model(img_a)
        emb_p = model(img_p)
        emb_n = model(img_n)
        # Calculate loss
        emb_loss = criterion(emb_a, emb_p, emb_n)

        # classify anchors
        out_a = model.classify(emb_a)
        # Calculate classification loss
        class_loss = classification_crit(out_a, label_a)

        loss = emb_loss + class_loss

        #print("ya", loss.item())
        loss.backward()
        optimizer.step()
        
        #total_loss += loss.item() * img_a.size(0)
        total_samples += img_a.size(0)
        emb_running_loss += emb_loss.item() * img_a.size(0)
        class_running_loss += class_loss.item() * img_a.size(0)

        # Predictions and Probabilities
        _, preds = torch.max(out_a, 1)
        probs = torch.softmax(out_a, dim=1)[:, 1] # Probability of class 1 (Melanoma)
        all_labels.extend(label_a.cpu().numpy())
        all_predictions.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().detach().numpy())

        #print("on est passe")

        if (i + 1) % 50 == 0:
            print(f'Batch {i+1}/{len(dataloader)}, Embedding training loss: {(emb_loss.item()):.4f}, Classification training loss: {(class_loss.item()):.4f}')
            #scheduler.step()

        if (i + 1) % 100 == 0:
            scheduler.step()

    emb_epoch_loss = emb_running_loss / total_samples
    class_epoch_loss = class_running_loss / total_samples
    acc = accuracy_score(all_labels, all_predictions)
    auc = roc_auc_score(all_labels, all_probs)
    aps = average_precision_score(all_labels, all_probs)
    return emb_epoch_loss, class_epoch_loss, acc, auc, aps

In [15]:
def evaluate(model, dataloader, criterion, classification_crit, device):
    """
    Evaluates the model by measuring inter-class distances.
    A simple approach: treat the model as a binary classifier using distance to a fixed 'Normal' centroid.
    NOTE: A more robust Siamese eval would use K-NN or a dedicated classification head.
    """
    model.eval()
    #embeddings = []
    #labels = []
    all_labels = []
    all_predictions = []
    all_probs = []
    emb_running_loss = 0.0
    class_running_loss = 0.0
    total_samples = 0
    
    with torch.no_grad():
        for _, (img_a, img_p, img_n, label_a) in enumerate(dataloader):

            #print("yo", i+1)
            img_a, img_p, img_n, label_a = img_a.to(device), img_p.to(device), img_n.to(device), label_a.to(device)
            
            # Get embeddings
            emb_a = model(img_a)
            emb_p = model(img_p)
            emb_n = model(img_n)
            # Calculate loss
            emb_loss = criterion(emb_a, emb_p, emb_n)

            # classify anchors
            out_a = model.classify(emb_a)
            # Calculate classification loss
            class_loss = classification_crit(out_a, label_a)
            
            #total_loss += loss.item() * img_a.size(0)
            total_samples += img_a.size(0)
            emb_running_loss += emb_loss.item() * img_a.size(0)
            class_running_loss += class_loss.item() * img_a.size(0)

            # Predictions and Probabilities
            _, preds = torch.max(out_a, 1)
            probs = torch.softmax(out_a, dim=1)[:, 1] # Probability of class 1 (Melanoma)
            all_labels.extend(label_a.cpu().numpy())
            all_predictions.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().detach().numpy())

    emb_epoch_loss = emb_running_loss / total_samples
    class_epoch_loss = class_running_loss / total_samples
    acc = accuracy_score(all_labels, all_predictions)
    auc = roc_auc_score(all_labels, all_probs)
    aps = average_precision_score(all_labels, all_probs)
    return emb_epoch_loss, class_epoch_loss, acc, auc, aps

In [8]:
20000//32//100


6

In [4]:
20000/6

3333.3333333333335

In [5]:
20000%6

2

In [ ]:
print(f"Using device: {device}")

train_samples, val_samples, test_samples = split_data(DATA_ROOT)
train_samples = train_samples.sample(frac=0.4).reset_index(drop=True)#frac=0.4
print(f"Number of normal samples in training data subset: {train_samples[train_samples["target"]== 0].shape[0]}")
print(f"Number of melanoma samples in training data subset: {train_samples[train_samples["target"]== 1].shape[0]}")

# Setup DataLoaders
train_dataset = TripletDataset(DATA_ROOT, train_samples,
                               transform=transforms.Compose([
                                transforms.RandomRotation(degrees=15, fill=(255, 255, 255)),
                                transforms.RandomHorizontalFlip(p=0.5),
                                transforms.RandomVerticalFlip(p=0.5),
                                transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05)
                                ]))

val_dataset = TripletDataset(DATA_ROOT, val_samples, transform=None)


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=True, num_workers=0)

# Setup Model, Loss, Optimizer
model = EmbeddingNet(out_dim=EMBEDDING_DIM).to(device)
#model = SiameseNet(embedding_net).to(device)
criterion = TripletLoss(margin=MARGIN).to(device)
classifier_crit = nn.CrossEntropyLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LEARNING_RATE, steps_per_epoch=train_samples.shape[0]//32//100,epochs=NUM_EPOCHS, anneal_strategy="cos")

# Training Loop
best_val_AP_score = -1.0
# metric logging
emb_train_loss_log = []
emb_val_loss_log = []
clas_train_loss_log = []
clas_val_loss_log = []
train_accuracy_log = []
val_accuracy_log = []
train_ROC_AUC_log = []
val_ROC_AUC_log = []
train_AP_score_log = []
val_AP_score_log = []

print("\n--- Starting Training ---")
for epoch in range(1, NUM_EPOCHS + 1):#NUM_EPOCHS
    # Train
    print(f"\n==== Training Epoch {epoch} ====")
    emb_train_loss, class_train_loss, train_acc, train_auc, train_aps = train_epoch(model, train_loader, criterion, classifier_crit, optimizer, scheduler, device)

    print(f"Epoch {epoch} training finished.")
    print(f"Average Training Embedding Loss: {emb_train_loss:.4f}")
    print(f"Average Training Classification Loss: {class_train_loss:.4f}")
    print(f"Training Classification Accuracy: {train_acc:.4f}")
    print(f"Training ROC AUC: {train_auc:.4f}")
    print(f"Training Average Precision Score: {train_aps:.4f}")

    emb_train_loss_log.append(emb_train_loss)
    clas_train_loss_log.append(class_train_loss)
    train_accuracy_log.append(train_acc)
    train_ROC_AUC_log.append(train_auc)
    train_AP_score_log.append(train_aps)
    
    # Evaluate
    emb_val_loss, class_val_loss, val_acc, val_auc, val_aps = evaluate(model, val_loader, criterion, classifier_crit, device)
    
    print("--- Validation phase ---")
    print(f"Average Validation Embedding Loss: {emb_val_loss:.4f}")
    print(f"Average Validation Classification Loss: {class_val_loss:.4f}")
    print(f"Validation Classification Accuracy: {val_acc:.4f}")
    print(f"Validation ROC AUC: {val_auc:.4f}")
    print(f"Validation Average Precision Score: {val_aps:.4f}")

    emb_val_loss_log.append(emb_val_loss)
    clas_val_loss_log.append(class_val_loss)
    val_accuracy_log.append(val_acc)
    val_ROC_AUC_log.append(val_auc)
    val_AP_score_log.append(val_aps)

    # Save best model
    if val_aps > best_val_AP_score:
        print(f"Previous best average precision score: {best_val_AP_score:.4f}")
        best_val_AP_score = val_aps
        print("Saving best model...")
        torch.save(model.state_dict(), (Path(DATA_ROOT) / 'best_siamese_model.pth'))
        
print("\n--- Training Finished ---")
print(f"Best Validation Average Precision Score: {best_val_AP_score:.4f}%")

Using device: cuda
Number of normal samples in training data subset: 1097
Number of melanoma samples in training data subset: 986

--- Starting Training ---

==== Training Epoch 1 ====
Batch 50/66, Embedding training loss: 1.0915, Classification training loss: 0.7137
Epoch 1 training finished.
Average Training Embedding Loss: 1.0358
Average Training Classification Loss: 0.6949
Training Classification Accuracy: 0.4911
Training ROC AUC: 0.5108
Training Average Precision Score: 0.4972
--- Validation phase ---
Average Validation Embedding Loss: 0.9838
Average Validation Classification Loss: 0.7001
Validation Classification Accuracy: 0.2137
Validation ROC AUC: 0.6267
Validation Average Precision Score: 0.0302
Previous best average precision score: -1.0000
Saving best model...

==== Training Epoch 2 ====
Batch 50/66, Embedding training loss: 1.0475, Classification training loss: 0.6907
Epoch 2 training finished.
Average Training Embedding Loss: 1.0173
Average Training Classification Loss: 0.

KeyboardInterrupt: 

In [35]:
train_samples, val_samples, test_samples = split_data(DATA_ROOT)

train_dataset = SkinDataset(DATA_ROOT, 
                            train_samples,
                            transform=transforms.Compose([
                                transforms.RandomRotation(degrees=15, fill=(255, 255, 255)),
                                transforms.RandomHorizontalFlip(p=0.5),
                                transforms.RandomVerticalFlip(p=0.5),
                                transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05)
                                ]))

val_dataset = SkinDataset(DATA_ROOT, val_samples, transform=None)

# Use standard DataLoader; Triplet mining is handled in the custom loss
train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, drop_last=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=VAL_TEST_BATCH_SIZE, shuffle=True, num_workers=0)

# Model Setup

# Embedding Net with Triplet Loss
embedding_net = EmbeddingNet(image_size=IMAGE_SIZE, out_dim=EMBEDDING_DIM).to(device)
embedding_crit = TripletMarginLoss(margin=MARGIN)
embedding_opt = optim.Adam(embedding_net.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(embedding_opt, mode='max', factor=0.5, patience=5)

# Classification Head with Cross-Entropy Loss
classifier_net = ClassificationNet(EMBEDDING_DIM).to(device)
classifier_crit = nn.CrossEntropyLoss().to(device)
classifier_opt = optim.Adam(classifier_net.parameters(), lr=LEARNING_RATE)# * 5) # Faster learning rate for small head

epochs = NUM_EPOCHS
#train_nets(
#        embedding_net, classifier_head, 
#        train_loader, val_loader, 
#        triplet_criterion, classification_criterion, 
#        embedding_optimizer, classifier_optimizer, 
#        embedding_scheduler,
#        NUM_EPOCHS, 
#        device)



print("\n--- Training Networks ---")

# metric logging intialisation
best_val_ROC_AUC = -1.0
emb_train_loss_log = []
clas_train_loss_log = []
train_accuracy_log = []
emb_val_loss_log = []
clas_val_loss_log = []
val_accuracy_log = []
val_ROC_AUC_log = []

for epoch in range(1, epochs + 1):
    embedding_net.train()
    classifier_net.train()
    emb_running_loss = 0.0
    clas_running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    print(f"\n==== Training Epoch {epoch} ====")

    # --- Training phase ----

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        #print(i)

        # ---- Embedding model training ----
        
        embedding_opt.zero_grad()
        classifier_opt.zero_grad()
        # Generate embeddings
        embeddings = embedding_net(images)
        
        # Calculate Triplet Loss using Batch-Hard mining
        emb_loss = embedding_crit(embeddings, labels)
        
        # ---- Classification model training ----
        
        
        # Classify
        outputs = classifier_net(embeddings)
        clas_loss = classifier_crit(outputs, labels)

        # Statistics
        emb_running_loss += emb_loss.item() * len(images)
        clas_running_loss += clas_loss.item() * len(images)
        _, preds = torch.max(outputs, 1)
        correct_predictions += torch.sum(preds == labels.data).item()
        total_samples += len(images)

        print(emb_loss.item(), emb_running_loss, clas_loss.item(), clas_running_loss)
        if (i + 1) % 50 == 0:
            print(f'Epoch {epoch}/{epochs}, Batch {i+1}/{len(train_loader)}, Embedding training loss: {(emb_loss.item()):.4f}, Classification training loss: {(clas_loss.item()):.4f}')
        #print(clas_loss)

        total_loss = emb_loss + clas_loss

        
        total_loss.backward()
        embedding_opt.step()
        classifier_opt.step()

    # embedding training epoch loss
    emb_epoch_loss = emb_running_loss / total_samples #len(train_loader.dataset)
    print(f"Epoch {epoch} finished. \nAverage Training Embedding Loss: {emb_epoch_loss:.4f}")

    # classification training epoch loss
    clas_epoch_loss = clas_running_loss / total_samples
    epoch_acc = correct_predictions / total_samples
    print(f"Average Training Classification Loss: {clas_epoch_loss:.4f}")
    print(f"Training Classification Accuracy: {epoch_acc:.4f}")

    # ---- Evaluation phase ----

    print("--- Validation phase ---")
    val_emb_loss, val_clas_loss, epoch_val_accuracy, epoch_val_ROC_AUC = evaluate_model(embedding_net, classifier_net, embedding_crit, classifier_crit, val_loader, device)
    print(f"Average Validation Embedding Loss: {val_emb_loss:.4f}")
    print(f"Average Validation Classification Loss: {val_clas_loss:.4f}")
    print(f"Validation Classification Accuracy: {epoch_val_accuracy:.4f}")
    print(f"Validation ROC AUC: {epoch_val_ROC_AUC:.4f}")

    # metric logging for plotting
    emb_train_loss_log.append(emb_epoch_loss)
    clas_train_loss_log.append(clas_epoch_loss)
    train_accuracy_log.append(epoch_acc)
    emb_val_loss_log.append(val_emb_loss)
    clas_val_loss_log.append(val_clas_loss)
    val_accuracy_log.append(epoch_val_accuracy)
    val_ROC_AUC_log.append(epoch_val_ROC_AUC)

    scheduler.step(epoch_val_ROC_AUC)

    # save best model based on ROC AUC
    if epoch_val_ROC_AUC > best_val_ROC_AUC:
        print(f"Previous best ROC AUC: {best_val_ROC_AUC:.4f}")
        best_val_ROC_AUC = epoch_val_ROC_AUC
        # Save model checkpoint
        print("Saving best model...")
        torch.save(embedding_net.state_dict(), Path(DATA_ROOT / 'best_embedding_model.pth'))
        torch.save(classifier_net.state_dict(), Path(DATA_ROOT / 'best_classifier_model.pth'))


print("Network training complete.")

# Graphical display of metric logs
plot_logs(
    emb_train_loss_log, 
    clas_train_loss_log,
    train_accuracy_log,
    emb_val_loss_log,
    clas_val_loss_log,
    val_accuracy_log,
    val_ROC_AUC_log,
    epochs)


--- Training Networks ---

==== Training Epoch 1 ====
1.4105521440505981 90.27533721923828 0.6941325664520264 44.42448425292969
nan nan nan nan
nan nan nan nan
nan nan nan nan
nan nan nan nan
nan nan nan nan
nan nan nan nan
nan nan nan nan
nan nan nan nan
nan nan nan nan
nan nan nan nan
nan nan nan nan
nan nan nan nan
nan nan nan nan
nan nan nan nan
nan nan nan nan
nan nan nan nan
nan nan nan nan
nan nan nan nan
nan nan nan nan
nan nan nan nan
nan nan nan nan
nan nan nan nan
nan nan nan nan
nan nan nan nan
nan nan nan nan
nan nan nan nan
nan nan nan nan
nan nan nan nan


KeyboardInterrupt: 

In [12]:
# metric logging intialisation
best_val_ROC_AUC = -1.0
emb_train_loss_log = []
clas_train_loss_log = []
train_accuracy_log = []
emb_val_loss_log = []
clas_val_loss_log = []
val_accuracy_log = []
val_ROC_AUC_log = []

In [13]:
embedding_net.train()
classifier_head.train()
emb_running_loss = 0.0
clas_running_loss = 0.0
correct_predictions = 0
total_samples = 0

In [14]:
for i, (images, labels) in enumerate(train_loader):
    images, labels = images.to(device), labels.to(device)
    if i == 0:
        break

print(images)
print(labels)

tensor([[[[2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 2.2489],
          [2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 2.2489],
          [2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 2.2489],
          ...,
          [2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 2.2489],
          [2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 2.2489],
          [2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 2.2489]],

         [[2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 2.4286],
          [2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 2.4286],
          [2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 2.4286],
          ...,
          [2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 2.4286],
          [2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 2.4286],
          [2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 2.4286]],

         [[2.6400, 2.6400, 2.6400,  ..., 2.6400, 2.6400, 2.6400],
          [2.6400, 2.6400, 2.6400,  ..., 2.6400, 2.6400, 2.6400],
          [2.6400, 2.6400, 2.6400,  ..., 2

In [29]:
embeddings = embedding_net(images)

In [30]:
images.shape

torch.Size([64, 3, 256, 256])

In [31]:
embeddings.shape

torch.Size([64, 128])

In [34]:
embeddings

tensor([[nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        ...,
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan]], device='cuda:0',
       grad_fn=<DivBackward0>)

In [33]:
emb_loss = triplet_criterion(embeddings, labels)
emb_loss

tensor(nan, device='cuda:0', grad_fn=<MeanBackward0>)

In [19]:
outputs = classifier_head(embeddings)
outputs

tensor([[ 0.0479,  0.0744],
        [ 0.0237,  0.0800],
        [ 0.0478,  0.0973],
        [ 0.0512,  0.0786],
        [ 0.0393,  0.0880],
        [ 0.0464,  0.0743],
        [ 0.0111,  0.0674],
        [ 0.0576,  0.0923],
        [ 0.0611,  0.0609],
        [ 0.0347,  0.0788],
        [ 0.0382,  0.0774],
        [ 0.0160,  0.0872],
        [ 0.0551,  0.0697],
        [ 0.0494,  0.0631],
        [ 0.0715,  0.1054],
        [-0.0108,  0.1196],
        [ 0.0299,  0.0603],
        [ 0.0239,  0.0684],
        [ 0.0194,  0.0501],
        [ 0.0075,  0.0995],
        [ 0.0670,  0.0818],
        [ 0.0594,  0.0578],
        [ 0.0174,  0.0901],
        [ 0.0020,  0.0933],
        [ 0.0479,  0.0772],
        [ 0.0612,  0.0842],
        [ 0.0027,  0.1050],
        [ 0.0487,  0.0947],
        [ 0.0326,  0.0844],
        [ 0.0397,  0.0907],
        [ 0.0308,  0.0724],
        [ 0.0506,  0.0829],
        [ 0.0544,  0.0757],
        [ 0.0558,  0.0805],
        [ 0.0114,  0.0966],
        [ 0.0361,  0

In [20]:
clas_loss = classification_criterion(outputs, labels)
clas_loss

tensor(0.6934, device='cuda:0', grad_fn=<NllLossBackward0>)

In [21]:
emb_running_loss += emb_loss.item() * len(images)
emb_running_loss

87.67748260498047

In [22]:
clas_running_loss += clas_loss.item() * len(images)
clas_running_loss

44.377403259277344

In [23]:
_, preds = torch.max(outputs, 1)
correct_predictions += torch.sum(preds == labels.data).item()
total_samples += len(images)
correct_predictions

29

In [24]:
print(f'Epoch , Batch {i+1}/{len(train_loader)}, Embedding training loss: {(emb_running_loss/total_samples):.4f}, Classification training loss: {(clas_running_loss/total_samples):.4f}')

Epoch , Batch 1/813, Embedding training loss: 1.3700, Classification training loss: 0.6934


In [25]:
total_loss = emb_loss + clas_loss
total_loss

tensor(2.0634, device='cuda:0', grad_fn=<AddBackward0>)

In [27]:
embedding_optimizer.zero_grad()
classifier_optimizer.zero_grad()
total_loss.backward()
embedding_optimizer.step()
classifier_optimizer.step()

In [28]:
for i, (images, labels) in enumerate(train_loader):
    images, labels = images.to(device), labels.to(device)
    if i == 0:
        break

print(images)
print(labels)

tensor([[[[ 2.2489,  2.2489,  2.2489,  ...,  2.2489,  2.2489,  2.2489],
          [ 2.2489,  2.2489,  2.2489,  ...,  2.2489,  2.2489,  2.2489],
          [ 2.2489,  2.2489,  2.2489,  ...,  2.2489,  2.2489,  2.2489],
          ...,
          [ 2.2489,  2.2489,  2.2489,  ...,  2.2489,  2.2489,  2.2489],
          [ 2.2489,  2.2489,  2.2489,  ...,  2.2489,  2.2489,  2.2489],
          [ 2.2489,  2.2489,  2.2489,  ...,  2.2489,  2.2489,  2.2489]],

         [[ 2.4286,  2.4286,  2.4286,  ...,  2.4286,  2.4286,  2.4286],
          [ 2.4286,  2.4286,  2.4286,  ...,  2.4286,  2.4286,  2.4286],
          [ 2.4286,  2.4286,  2.4286,  ...,  2.4286,  2.4286,  2.4286],
          ...,
          [ 2.4286,  2.4286,  2.4286,  ...,  2.4286,  2.4286,  2.4286],
          [ 2.4286,  2.4286,  2.4286,  ...,  2.4286,  2.4286,  2.4286],
          [ 2.4286,  2.4286,  2.4286,  ...,  2.4286,  2.4286,  2.4286]],

         [[ 2.6400,  2.6400,  2.6400,  ...,  2.6400,  2.6400,  2.6400],
          [ 2.6400,  2.6400,  

In [ ]:
def train_nets(
        embedding_net, classifier_net, 
        train_loader, val_loader, 
        embedding_crit, classifier_crit, 
        embedding_opt, classifier_opt, 
        scheduler,
        epochs, 
        device):
    """Trains the Siamese Embedding Network using Triplet Loss."""
    
    print("\n--- Training Networks ---")

    # metric logging intialisation
    best_val_ROC_AUC = -1.0
    emb_train_loss_log = []
    clas_train_loss_log = []
    train_accuracy_log = []
    emb_val_loss_log = []
    clas_val_loss_log = []
    val_accuracy_log = []
    val_ROC_AUC_log = []
    
    for epoch in range(1, epochs + 1):
        embedding_net.train()
        classifier_net.train()
        emb_running_loss = 0.0
        clas_running_loss = 0.0
        correct_predictions = 0
        total_samples = 0

        print(f"\n==== Training Epoch {epoch} ====")

        # --- Training phase ----

        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            #print(i)

            # ---- Embedding model training ----
            
            
            # Generate embeddings
            embeddings = embedding_net(images)
            
            # Calculate Triplet Loss using Batch-Hard mining
            emb_loss = embedding_crit(embeddings, labels)
            
            # ---- Classification model training ----
            
            
            # Classify
            outputs = classifier_net(embeddings)
            clas_loss = classifier_crit(outputs, labels)

            # Statistics
            emb_running_loss += emb_loss.item() * len(images)
            clas_running_loss += clas_loss.item() * len(images)
            _, preds = torch.max(outputs, 1)
            correct_predictions += torch.sum(preds == labels.data).item()
            total_samples += len(images)

            if (i + 1) % 50 == 0:
                print(f'Epoch {epoch}/{epochs}, Batch {i+1}/{len(train_loader)}, Embedding training loss: {(emb_running_loss/total_samples):.4f}, Classification training loss: {(clas_running_loss/total_samples):.4f}')
            #print(clas_loss)

            total_loss = emb_loss + clas_loss

            embedding_opt.zero_grad()
            classifier_opt.zero_grad()
            total_loss.backward()
            embedding_opt.step()
            classifier_opt.step()

        # embedding training epoch loss
        emb_epoch_loss = emb_running_loss / total_samples #len(train_loader.dataset)
        print(f"Epoch {epoch} finished. \nAverage Training Embedding Loss: {emb_epoch_loss:.4f}")

        # classification training epoch loss
        clas_epoch_loss = clas_running_loss / total_samples
        epoch_acc = correct_predictions / total_samples
        print(f"Average Training Classification Loss: {clas_epoch_loss:.4f}")
        print(f"Training Classification Accuracy: {epoch_acc:.4f}")

        # ---- Evaluation phase ----

        print("--- Validation phase ---")
        val_emb_loss, val_clas_loss, epoch_val_accuracy, epoch_val_ROC_AUC = evaluate_model(embedding_net, classifier_net, embedding_crit, classifier_crit, val_loader, device)
        print(f"Average Validation Embedding Loss: {val_emb_loss:.4f}")
        print(f"Average Validation Classification Loss: {val_clas_loss:.4f}")
        print(f"Validation Classification Accuracy: {epoch_val_accuracy:.4f}")
        print(f"Validation ROC AUC: {epoch_val_ROC_AUC:.4f}")

        # metric logging for plotting
        emb_train_loss_log.append(emb_epoch_loss)
        clas_train_loss_log.append(clas_epoch_loss)
        train_accuracy_log.append(epoch_acc)
        emb_val_loss_log.append(val_emb_loss)
        clas_val_loss_log.append(val_clas_loss)
        val_accuracy_log.append(epoch_val_accuracy)
        val_ROC_AUC_log.append(epoch_val_ROC_AUC)

        scheduler.step(epoch_val_ROC_AUC)

        # save best model based on ROC AUC
        if epoch_val_ROC_AUC > best_val_ROC_AUC:
            print(f"Previous best ROC AUC: {best_val_ROC_AUC:.4f}")
            best_val_ROC_AUC = epoch_val_ROC_AUC
            # Save model checkpoint
            print("Saving best model...")
            torch.save(embedding_net.state_dict(), Path(DATA_ROOT / 'best_embedding_model.pth'))
            torch.save(classifier_net.state_dict(), Path(DATA_ROOT / 'best_classifier_model.pth'))


    print("Network training complete.")

    # Graphical display of metric logs
    plot_logs(
        emb_train_loss_log, 
        clas_train_loss_log,
        train_accuracy_log,
        emb_val_loss_log,
        clas_val_loss_log,
        val_accuracy_log,
        val_ROC_AUC_log,
        epochs)

In [ ]:
# Evaluation
test_dataset = SkinDataset(DATA_ROOT, test_samples, transform=None)
val_loader = DataLoader(test_dataset, batch_size=VAL_TEST_BATCH_SIZE, shuffle=True, num_workers=0)
evaluate_model(embedding_net, classifier_head, val_loader, device)

In [29]:
512*6+241

3313

In [28]:
len(val_dataset)

3313